# Cboe 短期限风险：Stage1 上限局部价值来源门

仅将上一原生交易日 VIX9D/VIX3M 收盘比映射到2018–2022 OOF 决策日，避免指数推出前回算段；不读取局部价值标签。

In [1]:
import hashlib,json,os,pathlib,shlex
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/stage1_cboe_vix9d_vix3m_cap_source_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
for key in ('VIX9D','VIX3M'):assert sha(root/src['raw_'+key])==src['raw_'+key+'_sha256']
assert sha(root/src['source_manifest'])==src['source_manifest_sha256'] and sha(root/src['Qlib_provider']/'qlib_store_manifest.json')==src['Qlib_manifest_sha256']
raw_manifest=json.loads((root/src['source_manifest']).read_text());assert all(raw_manifest['sources'][key]['date_range'][0]<'2018-01-01' for key in ('VIX9D','VIX3M'))
source_binding=json.loads((card/'source_nas_binding.json').read_text());assert source_binding['remote_hash_verified'] and len(source_binding['scoped_originals_verified'])==2
frame_card=root/src['saved_OOF_frame_card'];label_card=root/src['saved_cap_label_card'];frame_summary=json.loads((frame_card/'summary.json').read_text());label_summary=json.loads((label_card/'summary.json').read_text());assert sha(frame_card/'summary.json')==src['saved_OOF_frame_summary_sha256'] and sha(label_card/'summary.json')==src['saved_cap_label_summary_sha256']
for card_root,eid,rid,name,digest in [(frame_card,frame_summary['experiment_id'],src['saved_OOF_frame_recorder_id'],'inputs/source_frame',src['saved_OOF_frame_artifact_sha256']),(frame_card,frame_summary['experiment_id'],src['saved_OOF_frame_recorder_id'],'paths/reference_0',src['saved_OOF_path_artifact_sha256']),(label_card,label_summary['experiment_id'],label_summary['recorder_id'],'analysis/labels',src['saved_cap_label_artifact_sha256'])]:assert sha(card_root/'qlib_recorder'/eid/rid/'artifacts'/name)==digest
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['Qlib_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2018-10-11',end_time='2022-12-30',freq='day')).normalize();quotes=D.features(['VIX9D','VIX3M'],['$close'],start_time='2018-10-11',end_time='2022-12-30',freq='day')['$close'].unstack('instrument').reindex(calendar)
assert {'VIX9D','VIX3M'}.issubset(quotes.columns)
with R.uri_context(str(frame_card/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=frame_summary['experiment_id'],recorder_id=src['saved_OOF_frame_recorder_id']);assert rec.status=='FINISHED';frame=rec.load_object('inputs/source_frame').reset_index(drop=True);path=rec.load_object('paths/reference_0')['replay'].reset_index(drop=True)
with R.uri_context(str(label_card/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=label_summary['experiment_id'],recorder_id=label_summary['recorder_id']);assert rec.status=='FINISHED';labels=rec.load_object('analysis/labels').reset_index(drop=True)
assert len(frame)==len(path)==len(labels)==1061 and frame.trade_date.equals(path.trade_date) and frame.trade_date.equals(labels.trade_date)
assert labels.loc[labels.eligible,'raw_beta_minus_cap'].gt(1e-8).all()
values=[];source_dates=[]
for signal in pd.DatetimeIndex(frame.signal_date).normalize():
 pos=int(calendar.searchsorted(signal));assert pos<len(calendar) and calendar[pos]==signal and pos>=1
 prior=calendar[pos-1];pair=quotes.loc[prior,['VIX9D','VIX3M']].to_numpy(float)
 values.append(float(np.log(pair[0]/pair[1])) if np.isfinite(pair).all() and (pair>0).all() else np.nan);source_dates.append(str(prior.date()))
mapped=pd.DataFrame({'signal_date':pd.to_datetime(frame.signal_date),'trade_date':pd.to_datetime(frame.trade_date),'source_session':source_dates,'log_VIX9D_over_VIX3M_prev':values,'known':np.isfinite(values),'eligible':labels.eligible.astype(bool),'cap_binding':labels.raw_beta_minus_cap.gt(1e-8).astype(bool),'label_available_at':pd.to_datetime(labels.label_available_at,utc=True),'alpha_probability':frame.alpha_probability.astype(float),'tail_probability':frame.tail_probability.astype(float),'raw_beta_minus_cap':labels.raw_beta_minus_cap.astype(float),'risk_state':path.risk_state.astype(float)})
assert mapped.signal_date.is_unique and (pd.to_datetime(mapped.source_session)<mapped.signal_date).all()
train=mapped.trade_date.dt.year.le(2020)&mapped.eligible&mapped.cap_binding&mapped.label_available_at.lt(pd.Timestamp('2021-01-01T00:00:00',tz='America/New_York').tz_convert('UTC'))
eval=mapped.trade_date.dt.year.ge(2021)&mapped.trade_date.dt.year.le(2022)&mapped.eligible&mapped.cap_binding&mapped.label_available_at.lt(pd.Timestamp('2023-01-01T00:00:00',tz='America/New_York').tz_convert('UTC'))
assert int(train.sum())==239 and int(eval.sum())==301 and int((eval&mapped.trade_date.dt.year.eq(2021)).sum())==56 and int((eval&mapped.trade_date.dt.year.eq(2022)).sum())==245
known_train=mapped.loc[train&mapped.known];features=['alpha_probability','tail_probability','raw_beta_minus_cap','risk_state'];corr={name:float(known_train[name].corr(known_train.log_VIX9D_over_VIX3M_prev,method='spearman')) for name in features}
static=pd.concat({'feature':mapped[['log_VIX9D_over_VIX3M_prev']],'meta':mapped[['known','eligible','cap_binding','source_session']]},axis=1);static.index=pd.MultiIndex.from_arrays([mapped.trade_date,['SOXL_SOXX_CASH']*len(mapped)],names=['datetime','instrument']);dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':('2018-10-15','2020-12-31'),'evaluation':('2021-01-01','2022-12-30')});assert len(dataset.prepare('train',col_set=['feature','meta']))==558 and len(dataset.prepare('evaluation',col_set=['feature','meta']))==503
result={'decision':'conditional_Cboe_VIX9D_VIX3M_prior_session_source_map_cap_local_receiver_only','plan_sha256':sha(card/'plan.json'),'source_original_NAS_sha256':source_binding['archive_sha256'],'calendar_rows':len(calendar),'OOF_signal_days':len(mapped),'OOF_known_days':int(mapped.known.sum()),'OOF_unknown_dates':[str(x.date()) for x in mapped.loc[~mapped.known,'signal_date']],'binding_mature_train_rows':int(train.sum()),'binding_mature_eval_rows':int(eval.sum()),'binding_mature_train_source_known':int((train&mapped.known).sum()),'binding_mature_eval_source_known':int((eval&mapped.known).sum()),'training_Spearman_vs_baseline_features':corr,'prelaunch_backfill_consumed':False,'historical_first_public_or_revision_qualified':False,'market_outcome_values_read':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=dataset);rec.save_objects(artifact_path='analysis',mapped_source=mapped,result=result);rec.log_metrics(OOF_known=result['OOF_known_days'],mature_train_known=result['binding_mature_train_source_known'],mature_eval_known=result['binding_mature_eval_source_known'],model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);settings=_settings(root);remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256'];result.update(recorder_id=rid,experiment_id=eid,recorder_uri=str(card/'qlib_recorder'),NAS_Recorder_sha256=remote,NAS_remote_hash_verified=True);(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({k:v for k,v in result.items() if k!='OOF_unknown_dates'},ensure_ascii=False,indent=2))


{
  "decision": "conditional_Cboe_VIX9D_VIX3M_prior_session_source_map_cap_local_receiver_only",
  "plan_sha256": "52f841364d4b60c2b9c456ab9972ca496403e90c74e0b352db92fbe8bf5b2712",
  "source_original_NAS_sha256": "90356bcc9c9b889e61464ae052f5845851ed9d10cdc784dbf4bf8572f597541e",
  "calendar_rows": 1063,
  "OOF_signal_days": 1061,
  "OOF_known_days": 1061,
  "binding_mature_train_rows": 239,
  "binding_mature_eval_rows": 301,
  "binding_mature_train_source_known": 239,
  "binding_mature_eval_source_known": 301,
  "training_Spearman_vs_baseline_features": {
    "alpha_probability": 0.3139218030308358,
    "tail_probability": 0.6045866882317781,
    "raw_beta_minus_cap": 0.01665113744242467,
    "risk_state": -0.08628791913356286
  },
  "prelaunch_backfill_consumed": false,
  "historical_first_public_or_revision_qualified": false,
  "market_outcome_values_read": 0,
  "model_fits": 0,
  "native_accounts": 0,
  "recorder_id": "a0812de7e96b4e668a70e8d017bc60b7",
  "experiment_id": "1241062

## 中文结论

Cboe VIX9D/VIX3M 上一原生会话比在1061/1061 OOF决策及239/301成熟受限标签上有值；不消费2014前回算。训练相关性尾部概率0.605、alpha0.314，不构成收益证据。原始CSV和原生Recorder均NAS冷读。历史首发／修订未证，旧H5版本不重开；0拟合0账户。